In [ ]:
from typing import List, TypedDict, Literal
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langgraph.graph import START, END, StateGraph
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_community.tools import TavilySearchResults
from langchain_core.output_parsers import PydanticOutputParser
load_dotenv()

In [ ]:
loader = PyPDFLoader("A:/langgraph_tutorial/Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf")
docs = loader.load()
len(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = splitter.split_documents(docs)

In [ ]:
len(chunks)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from pathlib import Path
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
if Path("faiss_index").exists():
    print('Loading local')
    vector_store = FAISS.load_local(
        "faiss_index",
        embedding_model,
        allow_dangerous_deserialization=True
    )
    print("Loaded existing FAISS index.")
else:
    vector_store = FAISS.from_documents(chunks, embedding_model)
    vector_store.save_local("faiss_index")
    print("Created and saved new FAISS index.")

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity',search_kwargs = {'k':4})

In [ ]:
retriever.invoke('what is machine learning')

In [ ]:
refine_llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash"
)
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=1000
)

In [ ]:
#State 
class State(TypedDict):
    query : str
    docs : List[Document]
    og_context : str
    score : float
    reason : str 
    verdict : Literal["CORRECT", "INCORRECT","AMBIGUOUS"]
    refined_context : str
    web_docs : List[Document] #web search context
    web_query : str
    answer : str



In [ ]:
def retrieve(state:State):
    q = state['query']
    return {'docs': retriever.invoke(q)}

In [ ]:
#Context Evaluator which gives verdict and refine context code

class ContextAssessment(BaseModel):
    score: float = Field(
        ge=0.0,
        le=1.0,
        description="Relevance and sufficiency of retrieved context for answering the query, from 0 to 1"
    )
    reason: str = Field(
        description="Explain why this score was assigned"
    )

evaluator_prompt = ChatPromptTemplate.from_template("""
You are an evaluator in a Corrective RAG system.

Evaluate how relevant and sufficient the retrieved context is
for answering the user's query.

Query:
{query}

Retrieved Context:
{context}

Instructions:
1. Give a relevance score between 0 and 1.
2. A higher score means the context is more relevant and sufficient.
3. Give score 0.5 if you think the context is ambiguous.
4. Give a low score if the context is irrelevant or does not contain
   enough information to answer the query.
5. Give a high score only when the context contains useful evidence
   that directly supports answering the query.


Return the structured output.
""")

evaluator_llm = llm.with_structured_output(ContextAssessment)

evaluator_chain = evaluator_prompt | evaluator_llm

def evaluate_context(state: State): 
    query = state['query']
    context = "\n\n".join(d.page_content for d in state['docs']).strip()

    result = evaluator_chain.invoke({'query': query, 'context': context})
    if result.score < 0.3:
        verdict = "INCORRECT"

    elif result.score > 0.7:
        verdict = "CORRECT" 
    else: 
        verdict = "AMBIGUOUS"
    return {
        'score': result.score,
        'reason' : result.reason, 
        'verdict' : verdict
    }




refinement_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a retrieval refinement component.

        Your task is to extract information relevant
        to the user's question.

        Rules:
        1. Keep only relevant information.
        2. Do not generate new information.
        3. Preserve the original meaning.
        4. Do not summarize unnecessarily.
        5. Preserve source IDs.
        6. If nothing is relevant, return NONE.

        Return only the refined context.
        """
    ),
    (
        "human",
        """
        Question:
        {query}

        Retrieved Context:
        {context}
        """
    )
])

refinement_chain = refinement_prompt | llm

def refine_context(state: State):

    query = state['query']
    if state['verdict'] == "CORRECT": 
        context = "\n\n".join(d.page_content for d in state['docs']).strip()
    elif state['verdict'] == "INCORRECT": 
        context = "\n\n".join(d.page_content for d in state['web_docs']).strip()
    else: 
        correct = "\n\n".join(d.page_content for d in state['docs']).strip()
        web_doc = "\n\n".join(d.page_content for d in state['web_docs']).strip()
        context = correct + "\n\n" + web_doc


    refined_context = refinement_chain.invoke({
        "query": query,
        "context": context
    })

    return {
        'og_context' : context,
        'refined_context': refined_context
    }






In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def generate(state:State):
    chain = prompt | llm

    out = chain.invoke({
        "question": state["query"],
        "context": state['refined_context']
    })

    return {"answer": out.content}

In [ ]:
#rewrire and web_docs code 
class WebQuery(BaseModel):
    query: str

parser = PydanticOutputParser(pydantic_object=WebQuery)

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user question into a web search query composed of keywords.\n"
            "Rules:\n"
            "- Keep it short (6–14 words).\n"
            "- If the question implies recency (e.g., recent/latest/last week/last month), add a constraint like (last 30 days).\n"
            "- Do NOT answer the question.\n"
            "- Return JSON with a single key: query",
        ),
        ("human", "Question: {question}"),
    ]
).partial(format_instructions=parser.get_format_instructions())


rewrite_chain = rewrite_prompt | llm | parser

def rewrite_query_node(state : State) -> State:
    out = rewrite_chain.invoke({"question": state['query']})
    return {'web_query' : out.query}


tavily = TavilySearchResults(max_results = 5)

def web_search_node(state: State) -> State:
    q = state['web_query'] or state['query']
    results = tavily.invoke({"query": q}) 

    web_docs = [] 
    for r in results or []: 
        title = r.get("title","")
        url = r.get("url","")
        content = r.get("content","") or r.get("snippet", "")
        text = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"

        web_docs.append(Document(page_content=text, metadata = {"url": url, "title":title}))

    return {
        'web_docs':web_docs
    }

    


def route_after_eval(state: State) -> str: 
    if state['verdict'] == "CORRECT":
        return "refine"
    else:
        return "rewrite_query"

In [ ]:
g = StateGraph(State)
g.add_node('retrieve',retrieve)
g.add_node('rewrite_query',rewrite_query_node)
g.add_node('evaluate_context', evaluate_context)
g.add_node('refine',refine_context)
g.add_node('generate',generate)
g.add_node('websearch', web_search_node)
g.add_edge(START, 'retrieve')
g.add_edge('retrieve','evaluate_context')
g.add_conditional_edges('evaluate_context',
                        route_after_eval,
                        {"refine":"refine", "rewrite_query":"rewrite_query"})
g.add_edge('rewrite_query','websearch')
g.add_edge('websearch','refine')
g.add_edge('refine','generate')
g.add_edge('generate',END)

app = g.compile()
app

In [ ]:
res = app.invoke({'query': 'What is Batch Normalization'})
print("verdict: ", res['verdict'])
print("reason: ", res['reason'])
print("web_query: ", res['web_query'])
print("output: ", res['answer'])
